In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import utils
import os
import json
import make_dag
from os.path import exists
# plt.style.use('./thesis.mplstyle')

In [ ]:
flux_name = "AIRS_mceq121_2011_sib_HG_E3"
crgamma = "0.25"
dir_base = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fits/FitToData/" + flux_name + "/crSlopeWidth_" + crgamma+ "/FS_0.50/"

In [ ]:
f =open(dir_base + "BestFitBlind.json")
BF=json.load(f)
f.close()

In [ ]:
llhs = np.load(dir_base + "/llhs.npy")

In [ ]:
arg_min = np.unravel_index(llhs[:,:,3].argmin(), llhs[:,:,3].shape)
dm2_min, Um4sq_min, Ut4sq_min, llh_min =llhs[arg_min]
th14_min, th24_min, th34_min = utils.Usq_to_thetas(Um4sq = Um4sq_min, Ut4sq = Ut4sq_min)
point_string = utils.get_point_string(dm2 = dm2_min, th24 = th24_min, th34 = th34_min, d24 = np.pi)

In [ ]:
assert(BF['LLH'] == llh_min)
assert(BF['Point'] == point_string)

In [ ]:
fluxdir_base= "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fluxes/" + flux_name + "/"
fluxdir = fluxdir_base + "Ut4sq_{:.6f}".format(Ut4sq_min)
realization_loc = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fits/FitToData/"+flux_name+ "/crSlopeWidth_"+ crgamma +"/BlindFit/realizations/"

cmd = "python3 -u /home/adiaz/IC-MEOWS-th34/Brazil.py  \
--point {point} \
--use_fastMC True \
--Emax 9976 \
--sim Platinum_Split_Plus_Tau \
--max_sim -1 \
--fast_scaling 0.50 \
--fastmc_dir /data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/FastMC/ \
--years 7.634 \
--fluxdir {fluxdir} \
--gen_realization True \
--realization_loc {realization_loc} \
--inject_fit True \
--inject_fit_file {inject_fit_file}".format(
point = point_string, fluxdir = fluxdir, realization_loc = realization_loc, \
inject_fit_file = dir_base + "BestFitBlind.json")



In [ ]:
print(cmd)
os.system(cmd)

In [ ]:
def make_fit_dag(n_seeds = "-1"):
    
    dag_filename ='./blindfit_n_seeds_{}.dag'.format(n_seeds,n_seeds)
    with open(dag_filename, 'w') as f:
        for i in range(2000):

            realization_file = realization_loc + "/Platinum_Split_Plus_Tau_7.634_" + flux_name + "/Ut4sq_{Ut4sq:.6f}/{point}/Realization_{i}.npz".format(
                Ut4sq = Ut4sq_min, point = point_string, i=i)
            dirname = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fits/FitToData/"+flux_name+ "/crSlopeWidth_"+ crgamma +"/BlindFit/"+"/n_seeds_{}/".format(n_seeds)
            m = make_dag.DagMaker(fast_scaling = '0.50', realization_file = realization_file,
                                          osg = 'True', do_fit = True, n_seeds = n_seeds,crSlopeWidth = "0.25")

            Ue4sq= 0.
            del14=0.

            m.addtodag(dm2=dm2_min, Ue4sq=Ue4sq, Um4sq=Um4sq_min, Ut4sq=Ut4sq_min, del14=del14,        
                       del24=np.pi, f=f, 
                       outdir = dirname,
                       job_index = i)



In [ ]:
make_fit_dag("-1")
make_fit_dag("4")